In [2]:
!pip install -q faiss-cpu sentence-transformers ujson openai


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [3]:
import torch, faiss, ujson as json
from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

/home/mmk2266/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ---- paths ----
EMB_DIR = Path("data/rag_embeddings")
INDEX_PATH = Path("data/rag.index")

In [5]:
# ---- Load FAISS index + metadata ----
index = faiss.read_index(str(INDEX_PATH))
meta = []
for jf in sorted(EMB_DIR.glob("*.jsonl")):
    with jf.open() as f:
        for line in f:
            meta.append(json.loads(line))
print(f"Loaded {len(meta)} chunks from {len(list(EMB_DIR.glob('*.jsonl')))} files.")

Loaded 5077 chunks from 23 files.


In [6]:
# ---- Load embedding model (for query encoding) ----
encoder = SentenceTransformer("Alibaba-NLP/gte-large-en-v1.5", trust_remote_code=True)

In [7]:
def retrieve(query, k=10):
    q_vec = encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, idxs = index.search(q_vec, k)
    results = []
    for score, i in zip(scores[0], idxs[0]):
        item = meta[int(i)]
        results.append({"score": float(score), **item})
    return results


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

In [9]:
MODEL_ID = "microsoft/phi-3-mini-4k-instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
    low_cpu_mem_usage=True,
)

Loading checkpoint shards: 100%|██████████| 2/2 [00:38<00:00, 19.27s/it]


In [32]:
SYSTEM = (
  "You are the Einstein of our generation, the best mathematician, philosopher and thinker. Synthesize an answer using the provided context only.\n"
  "Do NOT copy sentences verbatim. Quote at most short phrases (<10 words).\n"
  "Cite sources inline as [DOC:doc_id]. If unknown, say 'Not found in the given context.'"
)

def build_context(chunks, max_chars=900):
    blocks = []
    for c in chunks:
        text = (c.get("text_for_embedding") or c.get("text") or "")[:max_chars]
        blocks.append(f"<chunk doc_id='{c.get('doc_id')}' page='{c.get('page')}' type='{c.get('type')}'>\n{text}\n</chunk>")
    return "\n".join(blocks)

In [33]:
def generate_answer(query, context, max_new_tokens=220):
    prompt = (
        f"<task>\nQuestion: {query}\n\n"
        "Use the <chunk> blocks below. Paraphrase; avoid copying.\n"
        "Cite as [DOC:doc_id].\n</task>\n\n"
        f"<context>\n{context}\n</context>\n\n<answer>"
    )
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.5,
        top_p=0.9,
        do_sample=True,
        #repetition_penalty=1.15,
        no_repeat_ngram_size=8,
        eos_token_id=tok.eos_token_id,
    )
    text = tok.decode(out[0], skip_special_tokens=True)
    # Clip anything after our closing tag if the model parrots context
    return text.split("</answer>")[0].split("<answer>")[-1].strip()

In [34]:
#query = "How do hyperparameters affect scaling?"
#query = "What do scaling laws say about how to choose model size and training compute?"
query = "What methods exist to accelerate token generation during inference without retraining the language model?"
chunks = retrieve(query, k=10)
context = build_context(chunks, max_chars=200) 

In [35]:
print(generate_answer(query, context))

Speculative execution and hardware optimizations are two methods to speed up token generation during inference in large language models. Speculative execution, as proposed by Leviathan et al., Zhang et al., and He et al., involves using additional predictive decoding heads that enable the model to generate multiple potential outputs in parallel. This approach can significantly reduce the time required for token generation, especially in cases where the model can predict the next token with a high degree of confidence. On the other hand, hardware optimizations focus on improving the efficiency of the model's inference process by leveraging specialized hardware or software techniques that can accelerate the computation of token generation. This can include optimizing the model's memory usage, using specialized processors or accelerators, or employing techniques such as token dropping and KV cache compression to reduce the amount of data that needs to be processed during inference.

Refer

In [36]:
for i, ch in enumerate(chunks, 1):
    print(f"--- Chunk {i} | Score: {ch['score']:.3f} | Doc: {ch['doc_id']} | Page: {ch.get('page')} ---")
    print(ch.get("text_for_embedding", ch.get("text", ""))[:800])  # limit to 800 chars for readability
    print()


--- Chunk 1 | Score: 0.779 | Doc: 2312 | Page: 20 ---
[SECTION] Efficient Inference for Large Language Models. [PAGE] 20
[PARAGRAPH]
Speculative Execution. Speculative decoding (Leviathan et al., 2022; Zhang et al., 2023b; He et al., 2023) is a technique that uses a draft model for generation and uses the larger model to verify those tokens. This technique is orthogonal to us and can be used for further improvement. In the case of speculative decoding, the window in our method is updated with multiple tokens rather than one.

--- Chunk 2 | Score: 0.753 | Doc: 2211 | Page: 1 ---
[SECTION] Abstract [PAGE] 1
[PARAGRAPH]
Inference from large autoregressive models like Transformers is slow - decoding K tokens takes K serial runs of the model. In this work we introduce speculative decoding - an algorithm to sample from autoregressive models faster without any changes to the outputs , by computing several tokens in parallel. At the heart of our approach lie the observations that (1) hard lang